# Reproduce Experiment 3 — Hayden–Preskill recovery

Retrieves job `d7gls13jne2c73932620` and recomputes the recovery fidelity from the single-qubit tomography (⟨X⟩, ⟨Y⟩, ⟨Z⟩) of the scrambled (A) and decoded (B) circuits.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np

service = QiskitRuntimeService(channel="ibm_cloud")

def check(name, value, paper, tol):
    ok = abs(value - paper) <= tol
    print(f"{name:32s} recomputed {value:9.4f}   paper {paper:9.4f}   {'OK' if ok else 'DIFFERENT'}")

In [ ]:
from qiskit.quantum_info import DensityMatrix, random_statevector, state_fidelity

job = service.job("d7gls13jne2c73932620")
r = job.result()
Ax, Ay, Az, Bx, By, Bz = [float(r[i].data.evs) for i in range(6)]

X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Z = np.array([[1, 0], [0, -1]])
rho = lambda x, y, z: DensityMatrix((np.eye(2) + x*X + y*Y + z*Z) / 2)
psi = random_statevector(2, seed=99)

F_A = state_fidelity(psi, rho(Ax, Ay, Az), validate=False)
F_B = state_fidelity(psi, rho(Bx, By, Bz), validate=False)

In [ ]:
check("F scrambled", F_A, 0.530, 0.0005)
check("F recovered", F_B, 0.9983, 0.00005)